## Deep Learning for Environmental Impact Analysis

**Student:** Serhiienko Olena  
**Group:** IP-42

In this laboratory work, I investigate the use of deep learning for environmental impact analysis on the basis of historical meteorological observations. The practical part of the study is focused on forecasting daily maximum air temperature with a Long Short-Term Memory (LSTM) neural network and interpreting the obtained results for climate-related decision support.


## Aim of the Work

The aim of this work is to build and evaluate a deep learning model for forecasting daily maximum temperature from historical weather data and to demonstrate the practical value of such models for environmental monitoring and risk assessment.


## Research Objectives

The objectives of this laboratory work are as follows:

1. Obtain historical meteorological observations from the Open-Meteo archive.
2. Prepare the dataset for recurrent neural network modelling.
3. Train an LSTM model using temperature and solar radiation indicators.
4. Evaluate prediction accuracy with MAE, RMSE, and $R^2$.
5. Interpret the forecasting results from the perspective of heat-related environmental risk.

The selected variables, namely daily maximum temperature and daily shortwave radiation, make it possible to illustrate how recurrent neural networks capture temporal dependencies in environmental time series.


## 1. Configuration and Library Import

In [17]:
LATITUDE = 50.45
LONGITUDE = 30.52

LOOKBACK_DAYS = 14
START_DATE = "2010-01-01"
END_DATE = "2025-12-31"

DISPLAY_START_DATE = "2025-01-01"
DISPLAY_END_DATE = "2025-12-31"

DISPLAY_FORECAST_START_DATE = "2025-08-01"
DISPLAY_FORECAST_END_DATE = "2025-08-31"

HEATWAVE_WARNING = 27.0
CRITICAL_EMERGENCY = 31.0

print(f"Study coordinates: latitude={LATITUDE}, longitude={LONGITUDE}")
print(f"Sequence length: {LOOKBACK_DAYS} days")
print(f"Observation period: {START_DATE} to {END_DATE}")

Study coordinates: latitude=50.45, longitude=30.52
Sequence length: 14 days
Observation period: 2010-01-01 to 2025-12-31


In [18]:
import os
import random

import numpy as np
import pandas as pd
import plotly.express as px
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.models import Sequential

In [19]:
def set_seeds(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds(42)

available_gpus = tf.config.list_physical_devices("GPU")
if available_gpus:
    print(f"GPU detected: {available_gpus[0].name}")
else:
    print("GPU was not detected. Computation will run on CPU.")

GPU was not detected. Computation will run on CPU.


## 2. Data Acquisition

Historical daily observations are retrieved from the Open-Meteo archive. The dataset contains the daily maximum air temperature and the daily sum of shortwave radiation. Missing observations are removed before modelling in order to preserve input consistency.

In [20]:
url = (
    "https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={LATITUDE}&longitude={LONGITUDE}&"
    f"start_date={START_DATE}&end_date={END_DATE}&"
    "daily=temperature_2m_max,shortwave_radiation_sum&"
    "timezone=auto&format=csv"
)

df = pd.read_csv(url, skiprows=3)
df = df.rename(
    columns={
        "time": "Date",
        "temperature_2m_max (°C)": "MaxTemp",
        "shortwave_radiation_sum (MJ/m²)": "SolarRad",
    }
)
df["Date"] = pd.to_datetime(df["Date"])
df = df[["Date", "MaxTemp", "SolarRad"]].dropna().reset_index(drop=True)

print(f"Number of observations: {len(df)}")
display(df.head())
display(df.describe())

Number of observations: 5844


,Date,MaxTemp,SolarRad
0,2010-01-01,-6.3,1.83
1,2010-01-02,-6.2,2.66
2,2010-01-03,-11.0,3.29
3,2010-01-04,-12.0,3.87
4,2010-01-05,-8.7,3.60


,Date,MaxTemp,SolarRad
count,5844,5844.000000,5844.000000
mean,2017-12-31 12:00:00,13.257888,11.970674
min,2010-01-01 00:00:00,-22.300000,0.340000
25%,2013-12-31 18:00:00,3.900000,4.000000
50%,2017-12-31 12:00:00,13.500000,10.965000
75%,2021-12-31 06:00:00,22.900000,19.510000
max,2025-12-31 00:00:00,36.300000,29.580000
std,NaN,10.930232,8.240199


In [21]:
fig_overview = px.line(
    df,
    x="Date",
    y=["MaxTemp", "SolarRad"],
    title="Historical Behaviour of the Selected Environmental Variables",
    labels={"value": "Observed value", "variable": "Indicator"},
)
fig_overview.update_layout(hovermode="x unified")
fig_overview.show()

## 3. Data Preparation for LSTM Modelling

Recurrent neural networks require sequential input data. Therefore, the series is split into training and testing segments without shuffling, normalized using the training subset only, and converted into sliding windows of fixed length. Each input sequence contains observations for the previous 14 days, while the target variable is the maximum temperature for the next day.

In [22]:
feature_columns = ["MaxTemp", "SolarRad"]
data_values = df[feature_columns].values

split_index = int(len(data_values) * 0.8)
train_data = data_values[:split_index]
test_data = data_values[split_index:]

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)


def create_sequences(dataset, lookback):
    X_seq, y_target = [], []
    for i in range(len(dataset) - lookback):
        X_seq.append(dataset[i : i + lookback])
        y_target.append(dataset[i + lookback, 0])
    return np.array(X_seq), np.array(y_target)


X_train, y_train = create_sequences(train_scaled, LOOKBACK_DAYS)
X_test, y_test = create_sequences(test_scaled, LOOKBACK_DAYS)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (4661, 14, 2)
y_train shape: (4661,)
X_test shape: (1155, 14, 2)
y_test shape: (1155,)


## 4. LSTM Model Construction and Training

The predictive model is a compact LSTM network with one recurrent layer, dropout regularization, and one output neuron for temperature regression. Early stopping is applied in order to prevent unnecessary training after convergence of the validation loss.

In [23]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(units=64, return_sequences=False),
    Dropout(0.2),
    Dense(1),
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,217 (67.25 KB)

 Trainable params: 17,217 (67.25 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0269 - mae: 0.1086 - val_loss: 0.0040 - val_mae: 0.0504
Epoch 2/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0071 - mae: 0.0667 - val_loss: 0.0039 - val_mae: 0.0499
Epoch 3/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0064 - mae: 0.0631 - val_loss: 0.0041 - val_mae: 0.0509
Epoch 4/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0062 - mae: 0.0624 - val_loss: 0.0037 - val_mae: 0.0487
Epoch 5/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0057 - mae: 0.0597 - val_loss: 0.0041 - val_mae: 0.0509
Epoch 6/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0058 - mae: 0.0606 - val_loss: 0.0038 - val_mae: 0.0491
Epoch 7/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0055 - mae: 0.0593 - val_loss: 0.0034 - val_mae: 0.0471
Epoch 8/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0053 - mae: 0.0577 - val_loss: 0.0034 - val_mae: 0.0471
Epoch 9/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - lo

In [25]:
history_df = pd.DataFrame(
    {
        "Epoch": history.epoch,
        "Training Loss": history.history["loss"],
        "Validation Loss": history.history["val_loss"],
    }
)

fig_history = px.line(
    history_df,
    x="Epoch",
    y=["Training Loss", "Validation Loss"],
    title="Training Dynamics of the LSTM Model",
    labels={"value": "Loss", "variable": "Dataset"},
)
fig_history.update_layout(hovermode="x unified")
fig_history.show()

## 5. Model Evaluation

After training, the predicted values are transformed back to the original physical scale. The model is then assessed using three standard regression metrics:

- Mean Absolute Error (MAE) reflects the average magnitude of prediction error.
- Root Mean Squared Error (RMSE) penalizes larger deviations more strongly.
- The coefficient of determination ($R^2$) measures how well the model explains variability in the observed data.

In [26]:
predictions_scaled = model.predict(X_test)

dummy_pred = np.zeros((len(predictions_scaled), len(feature_columns)))
dummy_actual = np.zeros((len(y_test), len(feature_columns)))

dummy_pred[:, 0] = predictions_scaled.flatten()
dummy_actual[:, 0] = y_test.flatten()

pred_inverse = scaler.inverse_transform(dummy_pred)[:, 0]
actual_inverse = scaler.inverse_transform(dummy_actual)[:, 0]
residuals = actual_inverse - pred_inverse

start_date_for_predictions = df["Date"].iloc[split_index + LOOKBACK_DAYS]
prediction_dates = pd.date_range(
    start=start_date_for_predictions,
    periods=len(actual_inverse),
    freq="D",
)

plot_df = pd.DataFrame(
    {
        "Date": prediction_dates,
        "Actual Temperature": actual_inverse,
        "Predicted Temperature": pred_inverse,
        "Residuals": residuals,
    }
)

mae = mean_absolute_error(actual_inverse, pred_inverse)
rmse = np.sqrt(mean_squared_error(actual_inverse, pred_inverse))
r2 = r2_score(actual_inverse, pred_inverse)

metrics_df = pd.DataFrame(
    {
        "Metric": ["MAE", "RMSE", "R2"],
        "Value": [mae, rmse, r2],
    }
)

display(metrics_df)

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


,Metric,Value
0,MAE,2.177400
1,RMSE,2.794592
2,R2,0.927817


In [27]:
print("Model evaluation summary")
print(f"MAE  = {mae:.2f} °C")
print(f"RMSE = {rmse:.2f} °C")
print(f"R2   = {r2:.3f}")

Model evaluation summary
MAE  = 2.18 °C
RMSE = 2.79 °C
R2   = 0.928


In [28]:
fig_forecast = px.line(
    plot_df,
    x="Date",
    y=["Actual Temperature", "Predicted Temperature"],
    title="Observed and Predicted Daily Maximum Temperature",
    labels={"value": "Temperature (°C)", "variable": "Series"},
)
fig_forecast.update_traces(
    patch={"line": {"dash": "dash", "width": 2}},
    selector={"name": "Actual Temperature"},
)
fig_forecast.update_traces(
    patch={"line": {"dash": "solid", "width": 2}},
    selector={"name": "Predicted Temperature"},
)

if DISPLAY_START_DATE and DISPLAY_END_DATE:
    initial_display_range = [DISPLAY_START_DATE, DISPLAY_END_DATE]
else:
    initial_display_range = [plot_df["Date"].min(), plot_df["Date"].max()]

fig_forecast.update_xaxes(rangeslider_visible=True, range=initial_display_range)
fig_forecast.update_layout(hovermode="x unified")
fig_forecast.show()

In [29]:
fig_scatter = px.scatter(
    plot_df,
    x="Actual Temperature",
    y="Predicted Temperature",
    title="Goodness of Fit: Actual versus Predicted Values",
)

fig_scatter.add_shape(
    type="line",
    x0=plot_df["Actual Temperature"].min(),
    y0=plot_df["Actual Temperature"].min(),
    x1=plot_df["Actual Temperature"].max(),
    y1=plot_df["Actual Temperature"].max(),
    line=dict(color="red", dash="dash"),
)
fig_scatter.show()

In [30]:
fig_residuals = px.scatter(
    plot_df,
    x="Date",
    y="Residuals",
    title="Residual Analysis over Time",
    labels={"Residuals": "Prediction error (°C)"},
)
fig_residuals.add_hline(y=0, line_dash="dash", line_color="red")
fig_residuals.update_layout(hovermode="x unified")
fig_residuals.show()

## 6. Decision-Support Interpretation

To connect the model with environmental management practice, the forecast is interpreted through threshold-based warning levels. Such a simplified decision scheme illustrates how a neural network forecast can support heat-related risk communication.

In [31]:
forecast_period_df = plot_df[
    (plot_df["Date"] >= DISPLAY_FORECAST_START_DATE)
    & (plot_df["Date"] <= DISPLAY_FORECAST_END_DATE)
].copy()

if forecast_period_df.empty:
    print("No forecast records are available for the selected interpretation period.")
else:
    forecast_period_df["Warning level"] = "Normal conditions"
    forecast_period_df.loc[
        forecast_period_df["Predicted Temperature"] >= HEATWAVE_WARNING,
        "Warning level",
    ] = "Heatwave warning"
    forecast_period_df.loc[
        forecast_period_df["Predicted Temperature"] >= CRITICAL_EMERGENCY,
        "Warning level",
    ] = "Critical emergency"

    decision_summary = forecast_period_df[[
        "Date",
        "Actual Temperature",
        "Predicted Temperature",
        "Warning level",
    ]].copy()
    display(decision_summary.head(15))
    display(decision_summary["Warning level"].value_counts().rename_axis("Category").reset_index(name="Days"))

,Date,Actual Temperature,Predicted Temperature,Warning level
1002,2025-08-01,24.3,22.071043,Normal conditions
1003,2025-08-02,26.4,25.308846,Normal conditions
1004,2025-08-03,27.0,26.482347,Normal conditions
1005,2025-08-04,28.1,26.517318,Normal conditions
1006,2025-08-05,29.0,27.150868,Heatwave warning
1007,2025-08-06,28.4,27.798398,Heatwave warning
1008,2025-08-07,23.0,27.051110,Heatwave warning
1009,2025-08-08,23.5,24.302073,Normal conditions
1010,2025-08-09,25.0,24.165898,Normal conditions
1011,2025-08-10,26.8,25.038026,Normal conditions


,Category,Days
0,Normal conditions,26
1,Heatwave warning,5


In [32]:
if forecast_period_df.empty:
    forecast_data = pd.DataFrame()
else:
    forecast_data = forecast_period_df.copy()
    forecast_data["lat"] = LATITUDE
    forecast_data["lon"] = LONGITUDE
    forecast_data["date_display"] = forecast_data["Date"].dt.strftime("%Y-%m-%d")

    color_map = {
        "Normal conditions": "blue",
        "Heatwave warning": "orange",
        "Critical emergency": "red",
    }

    fig_map = px.scatter_mapbox(
        forecast_data,
        lat="lat",
        lon="lon",
        color="Warning level",
        size="Predicted Temperature",
        hover_data={
            "Predicted Temperature": ':.1f',
            "Actual Temperature": ':.1f',
            "Date": '|%Y-%m-%d',
            "lat": False,
            "lon": False,
            "date_display": False,
        },
        color_discrete_map=color_map,
        animation_frame="date_display",
        mapbox_style="open-street-map",
        zoom=9,
        height=550,
        title="Spatial Interpretation of Heat-Related Forecast Levels",
    )
    fig_map.update_layout(mapbox_center={"lat": LATITUDE, "lon": LONGITUDE})
    fig_map.show()

/tmp/ipykernel_257111/3398108858.py:15: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(


## Conclusion

As a result of this laboratory work, I studied the practical application of deep learning methods in environmental analysis and implemented an LSTM model for daily maximum temperature forecasting. The obtained results confirmed that recurrent neural networks are suitable for analysing temporal dependencies in meteorological data and can provide sufficiently informative forecasts for applied environmental tasks.

In my opinion, the completed analysis demonstrates that deep learning is not only an effective forecasting instrument, but also a useful basis for decision support in the context of climate adaptation. The model can be used to identify potentially dangerous temperature periods, which makes this approach relevant for environmental monitoring, early warning, and sustainable planning.